<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/power/gas_turbine_emissions_and_process_power.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gas Turbine Emissions and Process Power

Connect NeqSim compressor shaft power to fuel gas demand and direct CO2 emissions using transparent screening factors.


## Setup

Install imports and define the gas stream.


In [ ]:
# Install NeqSim when running in a fresh Colab session.
try:
    import neqsim
except ImportError:
    %pip install neqsim

import json
import pandas as pd
import matplotlib.pyplot as plt
try:
    from neqsim import jneqsim
except ImportError:
    from neqsim import jNeqSim as jneqsim
from neqsim.thermo import TPflash

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
Stream = jneqsim.process.equipment.stream.Stream
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

def make_gas(temperature_c=25.0, pressure_bara=60.0):
    fluid = SystemSrkEos(273.15 + temperature_c, pressure_bara)
    fluid.addComponent("nitrogen", 0.01)
    fluid.addComponent("CO2", 0.02)
    fluid.addComponent("methane", 0.86)
    fluid.addComponent("ethane", 0.07)
    fluid.addComponent("propane", 0.03)
    fluid.addComponent("n-butane", 0.01)
    fluid.setMixingRule("classic")
    TPflash(fluid)
    fluid.initProperties()
    return fluid

## Calculate Compressor Power

Run a simple compression duty as the process power demand.


In [ ]:
Compressor = jneqsim.process.equipment.compressor.Compressor
fluid = make_gas(25.0, 55.0)
feed = Stream("Fuel gas process feed", fluid)
feed.setFlowRate(10000.0, "kg/hr")
compressor = Compressor("Export compressor", feed)
compressor.setOutletPressure(120.0)
process = ProcessSystem()
process.add(feed)
process.add(compressor)
process.run()
shaft_power_kW = compressor.getPower("kW")
shaft_power_kW


## Estimate Fuel and Emissions

The factors are screening assumptions; replace with site-specific gas turbine curves for project work.


In [ ]:
thermal_efficiency = 0.32
fuel_lhv_MJ_per_kg = 48.0
co2_factor_kg_per_kg_fuel = 2.75
annual_hours = 8000

fuel_kg_per_hr = shaft_power_kW * 3.6 / thermal_efficiency / fuel_lhv_MJ_per_kg
co2_tonnes_per_year = fuel_kg_per_hr * co2_factor_kg_per_kg_fuel * annual_hours / 1000.0
pd.DataFrame([{"shaft_power_kW": shaft_power_kW, "fuel_kg_per_hr": fuel_kg_per_hr, "co2_tonnes_per_year": co2_tonnes_per_year}])


## Sensitivity

Screen emissions sensitivity to gas turbine efficiency.


In [ ]:
rows = []
for efficiency in [0.28, 0.32, 0.36, 0.40]:
    fuel = shaft_power_kW * 3.6 / efficiency / fuel_lhv_MJ_per_kg
    rows.append({"thermal_efficiency": efficiency, "co2_tonnes_per_year": fuel * co2_factor_kg_per_kg_fuel * annual_hours / 1000.0})
sens = pd.DataFrame(rows)
ax = sens.plot(x="thermal_efficiency", y="co2_tonnes_per_year", marker="o", legend=False)
ax.set_xlabel("Gas turbine thermal efficiency [-]")
ax.set_ylabel("CO2 emissions [tonnes/year]")
ax.grid(True)
plt.show()
